# Quick Commerce Swarm Orchestrator

This notebook is the interview-friendly walkthrough of the project. It explains what the system does, how data flows, how distances are calculated, how the picker routes are built, and how the final API response is assembled.

Use this notebook as a speaking script: each section answers one likely interview question.


## 1. Project Goal

The goal is to simulate a quick-commerce warehouse order-processing pipeline. A customer types an order in natural language, the system converts it into SKU-level items, checks stock, plans pick routes, and returns a structured response with metrics, alerts, and recommendations.

The project demonstrates three things well:

- Natural-language order understanding
- Deterministic warehouse routing
- Clear data modeling and workflow orchestration


## 2. Folder Map

Only a few folders matter for understanding the project:

- `app/` - Next.js frontend dashboard and explanation pages
- `backend/app/` - FastAPI simulation, parsing, routing, and typed models
- `backend/tests/` - small tests that verify the workflow
- `Quick_Commerce_Interview_Learning_Notebook.ipynb` - end-to-end project explanation

Everything else in the workspace is either build output, environment files, or temporary tooling data.


## 3. High-Level Architecture

The system has a simple layered design:

1. The frontend collects an instruction and picker count.
2. The backend parses the instruction into structured line items.
3. The workflow checks inventory and computes routes.
4. The API sends back a typed response for the UI to render.

This is intentionally not a mysterious "black box agent". The behavior is explicit and inspectable.


In [ ]:
from pathlib import Path

backend_root = Path("backend/app")
for name in ["main.py", "agent_workflow.py", "services.py", "models.py", "data.py"]:
    print(f"{name}: {(backend_root / name).exists()}")


## 4. Data Sources

The project uses seeded in-memory data instead of a live database. That keeps the demo reproducible and easy to explain.

The seed data includes:

- A catalog of SKUs
- Physical storage locations
- Inventory counts
- Safety anomalies
- Recommendation hints

If this were production, the SQL schema in `backend/app/schema.sql` would be the persistence layer.


### Why seed the data?

Seeded data makes the demo deterministic. The same input order always produces the same parsed items, same distances, and same routes, which is ideal for interviews and testing.


In [ ]:
from textwrap import indent

print("Core seed files:")
for file in ["backend/app/data.py", "backend/app/schema.sql"]:
    print("-", file)


## 5. Data Model

The most important typed objects are defined in `backend/app/models.py`. They describe the exact shape of the request, route, and response.

Key objects:

- `SimulationRequest` - input instruction and picker count
- `ParsedOrderItem` - parsed SKU plus quantity and confidence
- `RouteStop` - one stop on a picker route
- `PickerRoute` - a full route for one picker
- `SimulationMetrics` - comparison metrics
- `SimulationResponse` - full response returned by the API


In [ ]:
from pathlib import Path

models_text = Path("backend/app/models.py").read_text(encoding="utf-8")
print(models_text.splitlines()[0:80])


## 6. Order Parsing

The order parser turns text like `2 bananas, 1 milk, and 3 eggs` into structured inventory items.

There are two parsing modes:

- Deterministic fallback parser in `services.py`
- Optional LangChain structured LLM parser in `agent_workflow.py`

The fallback is the default so the project works without any API keys. If an LLM is enabled, the system still validates the model output against the known catalog before routing.


### What the parser does

- Normalizes the instruction to lowercase
- Matches product aliases like `milk`, `eggs`, or `glass cleaner`
- Extracts quantities such as `2`, `three`, or `dozen`
- Maps every recognized item to a real SKU id
- Assigns a confidence score

This is a good interview point: language understanding is separated from warehouse logic.


In [ ]:
from backend.app.services import parse_instruction

example = parse_instruction("2 bananas, 1 milk, and 3 eggs")
[(item.sku_id, item.quantity, item.fragility_score, item.confidence) for item in example]


## 7. How Distances Are Calculated

Distance is calculated with Manhattan distance on a warehouse grid. That means movement is allowed horizontally and vertically, not diagonally.

Formula:

`|x1 - x2| + |y1 - y2|`

Why this works well here:

- Warehouses are aisle-based, so walking paths behave like grid movement
- It is simple, deterministic, and explainable
- It is easy to test and visualize


In [ ]:
from backend.app.services import manhattan

print(manhattan((0, 0), (3, 4)))
print(manhattan((2, 5), (7, 1)))


## 8. FIFO Route

The FIFO route is the baseline. It picks items in the same order they were parsed from the customer instruction.

This gives us a simple reference path so we can compare the optimized swarm strategy against something understandable.

Conceptually:

1. Start at dispatch
2. Visit each item in order
3. Return to dispatch
4. Sum all Manhattan legs


In [ ]:
from backend.app.services import fifo_route

items = parse_instruction("2 bananas, 1 milk, and 3 eggs")
route = fifo_route(items)
route.distance, [(s.sku_id, s.step) for s in route.stops]


## 9. Optimized Swarm Routing

The optimized strategy divides work across multiple pickers. The goal is not just to reduce total walking, but to reduce the longest critical path so the order can be completed faster.

The heuristic in `services.py` does the following:

- Sorts items by fragility
- Tries to balance route workload across the picker pool
- Chooses the next assignment by looking at projected route cost
- Uses the maximum picker distance as the main score

Interview translation: it is a deterministic multi-picker scheduler, not a learned model.


### Why fragility matters

Fragility score influences ordering so delicate items can be considered earlier in the routing logic. This is useful in a dark-store context because operational concerns are not only about walking distance; they also involve item handling.


In [ ]:
from backend.app.services import optimized_routes

optimized = optimized_routes(items, picker_count=2)
[(r.picker_id, r.distance, [(s.sku_id, s.step) for s in r.stops]) for r in optimized]


## 10. Workflow Orchestration

`backend/app/agent_workflow.py` uses LangGraph to make the workflow explicit. The graph has four nodes:

1. `parse_order`
2. `validate_stock`
3. `plan_routes`
4. `build_response`

This is valuable because each stage has a single responsibility and a typed input/output contract.


### Why use a graph here?

A graph is easier to explain than an opaque agent loop. Each step is visible, testable, and replaceable. If the parsing logic changes later, the route planner and response builder do not need to be rewritten.


In [ ]:
from backend.app.agent_workflow import SIMULATION_GRAPH

node_names = set(SIMULATION_GRAPH.get_graph().nodes)
node_names


## 11. Validation and Safety

After parsing, the workflow checks whether the requested quantity is actually in stock. It also carries forward seeded anomalies and placement recommendations.

This adds realism because a warehouse system is not only about route distance. It also needs operational safety checks and inventory awareness.


In [ ]:
from backend.app.agent_workflow import run_agent_simulation

response = run_agent_simulation("2 bananas, 1 milk, and 3 eggs", picker_count=2)
response.state, response.metrics.fifo_distance, response.metrics.optimized_distance, response.metrics.reduction_percent


## 12. Metrics

The response includes a compact set of metrics that are easy to explain in an interview:

- `fifo_distance` - baseline path length
- `optimized_distance` - longest picker route in the swarm
- `reduction_percent` - how much better the optimized route is
- `parser_confidence` - average confidence across parsed items
- `active_alerts` - number of seeded operational alerts
- `dispatch_seconds` - simulated dispatch time derived from route distance

The important idea is that the system returns measurable outcomes, not just a route string.


In [ ]:
response.metrics.model_dump()


## 13. End-to-End Flow

Here is the simplest possible explanation of the whole system:

- User writes a natural-language order
- Parser maps text to SKUs and quantities
- Stock checker flags shortages
- FIFO route is computed as the baseline
- Optimized routes are computed for multiple pickers
- Metrics and warehouse annotations are packaged into the API response
- Frontend displays the result visually

That is the full story from input to result.


## 14. What the Frontend Shows

The Next.js app is a presentation layer. It visualizes:

- Parsed items
- FIFO vs optimized route
- Picker progress
- Safety alerts
- Recommendation cards
- Backend health status

The UI is useful for demos, but the real logic lives in the backend.


## 15. How to Explain This In An Interview

A strong 30-second explanation is:

> This project simulates a quick-commerce warehouse. I parse a natural-language order into SKUs, validate stock, compute a FIFO baseline route, and then assign items across multiple pickers using a deterministic heuristic with Manhattan distance. The backend is orchestrated with a LangGraph workflow and returns typed metrics, safety alerts, and route results for the frontend to visualize.

If the interviewer asks for depth, talk about:

- Data modeling
- Deterministic fallback vs optional LLM parsing
- Manhattan distance
- Critical path optimization
- Why graph orchestration is easier to test


## 16. Suggested Improvements

If you want to extend the project later, good next steps are:

- Persist the seed data in PostgreSQL
- Add unit tests for route distance and parser edge cases
- Replace heuristics with a real optimization solver
- Add live stock mutation and order history
- Move explanation pages into a single documentation section

For now, the current version is intentionally simple enough to present clearly.
